# Information Hunter - Tinker SFT Training and Inference
This script fine-tunes `Qwen/Qwen3.5-4B` on the augmented SFT dataset
and runs batch inference on `evaluation_rag_queries_aro_BORO.csv` to extract verified evidence.
Powered by the Tinker API for remote GPU execution.


In [1]:
# ============================================================
# Cell 1: Configuration
# ============================================================

class Config:
    # Model configuration
    model_name = "Qwen/Qwen3.5-4B"
    thinking = False  # Set to True to enable model thinking/reasoning, False to disable
    
    # LoRA configuration
    lora_rank = 32
    
    # Optimizer & Scheduler
    lr = 2e-4
    epoch = 1
    batch_size = 16             # Effective batch size (equivalent to Unsloth training batch size)
    max_length = 2048*2            # Max sequence length matching Unsloth config
    warmup_steps = 10           # Number of warmup steps
    
    # Dataset Configurations
    max_samples = 5000            # Limit dataset size for fast iteration (-1 for full)
    eval_dataset_size = 50     # Validation split size matching Unsloth config
    seed = 42
    
    # Save options
    save_every = 500             # Checkpoint every 50 steps
    
    # Paths
    train_dataset_path = r"/kaggle/input/datasets/barnobarno/sssft-newwww/augmented_sft_dataset.csv"
    test_dataset_path = r"/kaggle/input/datasets/barnobarno/rag-retrievals-ready-for-inference/evaluation_rag_queries_aro_BORO.csv"
    output_dataset_path = r"evaluation_rag_queries_aro_BORO_tinker_predictions.csv"

print("Config loaded:")
for k, v in vars(Config).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")


Config loaded:
  model_name: Qwen/Qwen3.5-4B
  thinking: False
  lora_rank: 32
  lr: 0.0002
  epoch: 1
  batch_size: 16
  max_length: 4096
  warmup_steps: 10
  max_samples: 5000
  eval_dataset_size: 50
  seed: 42
  save_every: 500
  train_dataset_path: /kaggle/input/datasets/barnobarno/sssft-newwww/augmented_sft_dataset.csv
  test_dataset_path: /kaggle/input/datasets/barnobarno/rag-retrievals-ready-for-inference/evaluation_rag_queries_aro_BORO.csv
  output_dataset_path: evaluation_rag_queries_aro_BORO_tinker_predictions.csv


In [2]:
# ============================================================
# Cell 2: Install Dependencies (For Cloud Execution)
# ============================================================

!pip install -q tinker tinker-cookbook datasets transformers pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.0/235.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.6/934.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.43.0 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opente

In [3]:
# ============================================================
# Cell 3: Imports & API Key
# ============================================================

import os
import sys
import time
import random
import logging
import math
import pandas as pd
import numpy as np
import tinker

from datasets import Dataset
from tinker_cookbook import model_info, renderers, tokenizer_utils
from tinker_cookbook.renderers import TrainOnWhat, get_renderer, get_text_content
from tinker_cookbook.supervised.data import conversation_to_datum
from tinker.types import SamplingParams

# ---- SET YOUR TINKER API KEY HERE ----
# Replace the placeholder below with your actual API key


from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("TINKER")
os.environ["TINKER_API_KEY"] = secret_value_0
# if "TINKER_API_KEY" not in os.environ:
    

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("qwen-sft-tinker")

print(f"Tinker SDK version: {tinker.__version__}")
api_set = 'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'
print(f"API key set: {api_set}")


Tinker SDK version: 0.22.7
API key set: True


In [4]:
# ============================================================
# Cell 4: Load & Split Dataset
# ============================================================

def load_and_prepare_dataset():
    print(f"Loading training dataset from: {Config.train_dataset_path}")
    if not os.path.exists(Config.train_dataset_path):
        raise FileNotFoundError(f"Training dataset not found: {Config.train_dataset_path}")
        
    df = pd.read_csv(Config.train_dataset_path)
    print(f"Loaded {len(df)} samples.")
    
    # Trim / limit samples if Config.max_samples is set
    if Config.max_samples > 0 and len(df) > Config.max_samples:
        df = df.iloc[:Config.max_samples].reset_index(drop=True)
        print(f"Sampled down to first {len(df)} rows.")

    # Determine validation split size
    eval_size = Config.eval_dataset_size
    if len(df) <= eval_size:
        eval_size = max(1, int(len(df) * 0.1))
        print(f"Dataset size is smaller than eval_dataset_size. Adjusting validation split size to {eval_size}")
        
    # Convert to HuggingFace Dataset
    raw_dataset = Dataset.from_pandas(df)
    
    # Split into Train and Eval
    print(f"Splitting dataset into Train and Eval ({eval_size} validation rows)...")
    split_dataset = raw_dataset.train_test_split(test_size=eval_size, seed=Config.seed)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
    
    print(f"Train split size: {len(train_dataset)}")
    print(f"Eval split size: {len(eval_dataset)}")
    
    return train_dataset, eval_dataset

train_dataset, eval_dataset = load_and_prepare_dataset()


Loading training dataset from: /kaggle/input/datasets/barnobarno/sssft-newwww/augmented_sft_dataset.csv
Loaded 100003 samples.
Sampled down to first 5000 rows.
Splitting dataset into Train and Eval (50 validation rows)...
Train split size: 4950
Eval split size: 50


In [5]:
# ============================================================
# Cell 5: Initialize Tokenizer & Renderer
# ============================================================

tokenizer = tokenizer_utils.get_tokenizer(Config.model_name)
print(f"Tokenizer loaded! Vocab size: {tokenizer.vocab_size}")

# Select the appropriate renderer based on thinking configuration
renderer_name = "qwen3_5_disable_thinking" if not Config.thinking else "qwen3_5"
print(f"Selected renderer: {renderer_name}")

renderer = get_renderer(renderer_name, tokenizer)


2026-07-13 09:42:31,754 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json: 0.00B [00:00, ?B/s]

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizer loaded! Vocab size: 248044
Selected renderer: qwen3_5_disable_thinking


In [6]:
# ============================================================
# Cell 6: Convert All Rows to Tinker Datums
# ============================================================

def make_datum_from_row(row):
    # Formats SFT training inputs into standard chat format
    conversation = [
        {"role": "user", "content": str(row["input"]).strip()},
        {"role": "assistant", "content": str(row["output"]).strip()}
    ]
    datum = conversation_to_datum(
        conversation,
        renderer,
        train_on_what=TrainOnWhat.LAST_ASSISTANT_MESSAGE,
        max_length=Config.max_length
    )
    return datum

print("Converting Train conversations to Tinker Datums...")
train_datums = []
skipped_train = 0
for idx, row in enumerate(train_dataset):
    try:
        datum = make_datum_from_row(row)
        train_datums.append(datum)
    except Exception as e:
        skipped_train += 1
        if skipped_train <= 5:
            print(f"  ⚠ Skipped train index {idx}: {e}")

print("Converting Eval conversations to Tinker Datums...")
eval_datums = []
skipped_eval = 0
for idx, row in enumerate(eval_dataset):
    try:
        datum = make_datum_from_row(row)
        eval_datums.append(datum)
    except Exception as e:
        skipped_eval += 1
        if skipped_eval <= 5:
            print(f"  ⚠ Skipped eval index {idx}: {e}")

print(f"\nConverted {len(train_datums)} train datums ({skipped_train} skipped)")
print(f"Converted {len(eval_datums)} eval datums ({skipped_eval} skipped)")


2026-07-13 09:42:35,485 [INFO] Weight reduction: 'mean' (token-mean loss)


Converting Train conversations to Tinker Datums...
Converting Eval conversations to Tinker Datums...

Converted 4950 train datums (0 skipped)
Converted 50 eval datums (0 skipped)


In [7]:
# ============================================================
# Cell 7: Training Definition
# ============================================================

def compute_nll(fwd_bwd_result, batch):
    """Compute mean negative log-likelihood from forward-backward results."""
    total_nll = 0.0
    total_tokens = 0.0
    for output, datum in zip(fwd_bwd_result.loss_fn_outputs, batch):
        logprobs = output["logprobs"].to_torch()
        w = datum.loss_fn_inputs["weights"].to_torch()
        total_nll -= logprobs.dot(w).item()
        total_tokens += w.sum().item()
    return total_nll / max(total_tokens, 1.0)


def train_lora(train_datums, eval_datums):
    """Full LoRA SFT training loop on Tinker for Qwen3.5 4B."""
    cfg = Config
    
    n_batches_per_epoch = len(train_datums) // cfg.batch_size
    total_steps = n_batches_per_epoch * cfg.epoch
    warmup_steps = cfg.warmup_steps
    
    print(f"╔{'═'*58}╗")
    print(f"║  Qwen3.5 4B SFT Training")
    print(f"╠{'═'*58}╣")
    print(f"║  Train samples: {len(train_datums):>8}  │  Eval samples: {len(eval_datums):>5}")
    print(f"║  Batch size:    {cfg.batch_size:>8}  │  Steps/epoch:  {n_batches_per_epoch:>5}")
    print(f"║  Total steps:   {total_steps:>8}  │  Warmup steps: {warmup_steps:>5}")
    print(f"║  LR: {cfg.lr:.1e}  │  Thinking: {cfg.thinking}")
    print(f"║  LoRA rank: {cfg.lora_rank}  │  Save every: {cfg.save_every} steps")
    print(f"╚{'═'*58}╝")
    
    print("\n⏳ Connecting to Tinker API...")
    service_client = tinker.ServiceClient()
    training_client = service_client.create_lora_training_client(
        base_model=cfg.model_name,
        rank=cfg.lora_rank,
    )
    print("✓ Connected!\n")
    
    global_step = 0
    train_losses = []
    t_start = time.time()
    
    for epoch in range(cfg.epoch):
        epoch_indices = list(range(len(train_datums)))
        random.seed(cfg.seed + epoch)
        random.shuffle(epoch_indices)
        
        for batch_idx in range(n_batches_per_epoch):
            step_start = time.time()
            
            # LR Schedule: Linear warmup -> Linear decay
            if global_step < warmup_steps:
                lr_mult = global_step / warmup_steps
            else:
                lr_mult = max(0.0, 1.0 - (global_step - warmup_steps) / max(1, total_steps - warmup_steps))
            current_lr = cfg.lr * lr_mult
            
            adam_params = tinker.AdamParams(
                learning_rate=current_lr,
                beta1=0.9,
                beta2=0.95,
                eps=1e-8,
            )
            
            # Build current batch
            start_idx = batch_idx * cfg.batch_size
            batch = [train_datums[epoch_indices[i]] for i in range(start_idx, start_idx + cfg.batch_size)]
            
            # Forward + Backward + Optimizer Update
            fwd_bwd_future = training_client.forward_backward(batch, loss_fn="cross_entropy")
            optim_future = training_client.optim_step(adam_params)
            
            fwd_bwd_result = fwd_bwd_future.result()
            optim_result = optim_future.result()
            
            train_nll = compute_nll(fwd_bwd_result, batch)
            train_losses.append(train_nll)
            step_time = time.time() - step_start
            
            # Logging
            if global_step % 5 == 0 or global_step == total_steps - 1:
                elapsed = time.time() - t_start
                avg_loss = sum(train_losses[-10:]) / len(train_losses[-10:])
                print(f"[Step {global_step:4d}/{total_steps}] "
                      f"epoch={epoch+1}/{cfg.epoch} lr={current_lr:.2e} "
                      f"train_nll={train_nll:.4f} avg_nll(10)={avg_loss:.4f} "
                      f"step={step_time:.1f}s elapsed={elapsed:.0f}s")
            
            # Save periodic checkpoint
            if cfg.save_every > 0 and global_step % cfg.save_every == 0 and global_step > 0:
                name = f"step_{global_step:04d}"
                training_client.save_state(name=name).result()
                print(f"  💾 Saved checkpoint: {name}")
                
            global_step += 1
            
    # Save final weights
    print("\n⏳ Saving final checkpoint...")
    training_client.save_state(name="final").result()
    sampler_result = training_client.save_weights_for_sampler(name="qwen3_5_sft_final").result()
    sampler_path = sampler_result.path
    
    total_time = time.time() - t_start
    print(f"\n============================================================")
    print(f"  ✅ TRAINING COMPLETE!")
    print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
    print(f"  Final train NLL: {train_losses[-1]:.4f}")
    print(f"  Sampler path:    {sampler_path}")
    print(f"============================================================")
    
    return service_client, sampler_path


In [8]:
# ============================================================
# Cell 8: Inference Function Definition
# ============================================================

# SFT English prompt instructions matching original Unsloth pipeline
INSTRUCTION = (
    "You will be provided with a question and some relevant reference contexts. "
    "Using the provided contexts, extract the correct answer to the question verbatim. "
    "If the answer is not present in the contexts, output exactly '[তথ্য পাওয়া যায়নি]'. "
    "Do not include any extra explanations, details, or formatting."
)

def run_test_inference(service_client, sampler_path):
    """Run batch inference on the test set and save results."""
    from tqdm import tqdm
    
    test_csv_path = Config.test_dataset_path
    output_csv_path = Config.output_dataset_path
    
    print(f"Loading test dataset from: {test_csv_path}")
    test_df = pd.read_csv(test_csv_path)
    print(f"Loaded {len(test_df)} rows for evaluation.")
    
    print(f"Connecting to Tinker sampler with path: {sampler_path}...")
    sampling_client = service_client.create_sampling_client(model_path=sampler_path)
    
    stop_sequences = renderer.get_stop_sequences()
    # If thinking is enabled, we need more tokens to output reasoning before final answer
    max_gen_tokens = 1024 if Config.thinking else 256
    params = SamplingParams(max_tokens=max_gen_tokens, temperature=0.0, stop=stop_sequences)
    
    # Pre-allocate results array with None placeholders
    extracted_answers = [None] * len(test_df)
    
    # Identify rows that need inference vs. rows with missing context (skip directly)
    inference_indices = []
    for idx, row in test_df.iterrows():
        retrieved_context = row.get("retrieved_context")
        if pd.isna(retrieved_context) or str(retrieved_context).strip() == "" or str(retrieved_context).strip().upper() == "[NULL]":
            extracted_answers[idx] = "[তথ্য পাওয়া যায়নি]"
        else:
            inference_indices.append(idx)
            
    print(f"Skipped {len(test_df) - len(inference_indices)} rows (missing/empty context).")
    print(f"Submitting {len(inference_indices)} test inference jobs asynchronously to Tinker...")
    
    futures = []
    for idx in inference_indices:
        row = test_df.iloc[idx]
        question = str(row.get("prompt_bn", "")).strip()
        retrieved_context = str(row.get("retrieved_context", "")).strip()
        
        prompt_text = (
            f"Instruction: {INSTRUCTION}\n\n"
            f"Context:\n{retrieved_context}\n\n"
            f"Question: {question}\n"
            f"Answer:"
        )
        
        messages = [{"role": "user", "content": prompt_text}]
        prompt = renderer.build_generation_prompt(messages)
        
        # Async sample request
        future = sampling_client.sample(prompt, sampling_params=params, num_samples=1)
        futures.append((idx, future))
        
    print("Collecting inference results...")
    for idx, future in tqdm(futures, desc="Running SFT Inference"):
        try:
            output = future.result()
            response_msg = renderer.parse_response(output.sequences[0].tokens)
            if isinstance(response_msg, tuple):
                response_msg = response_msg[0]
                
            content = response_msg["content"]
            if isinstance(content, list):
                pred_text = "".join(p.get("text", "") for p in content if p["type"] == "text")
            else:
                pred_text = content
                
            # Strip out any remaining thinking blocks as a safety fallback
            if "</think>" in pred_text:
                pred_text = pred_text.split("</think>")[-1]
            elif "</thought>" in pred_text:
                pred_text = pred_text.split("</thought>")[-1]
            elif "</thinking>" in pred_text:
                pred_text = pred_text.split("</thinking>")[-1]
                
            extracted_answers[idx] = pred_text.strip()
        except Exception as e:
            print(f"Error predicting row index {idx}: {e}")
            extracted_answers[idx] = "[তথ্য পাওয়া যায়নি]"  # Fallback on failure
            
    # Save generated outputs to new column in the CSV
    test_df["ih_extracted_answer"] = extracted_answers
    
    # Create target parent directories if not exist
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    test_df.to_csv(output_csv_path, index=False)
    print(f"Successfully saved batch inference results to: {output_csv_path}")


In [9]:
service_client, sampler_path = train_lora(train_datums, eval_datums)

╔══════════════════════════════════════════════════════════╗
║  Qwen3.5 4B SFT Training
╠══════════════════════════════════════════════════════════╣
║  Train samples:     4950  │  Eval samples:    50
║  Batch size:          16  │  Steps/epoch:    309
║  Total steps:        309  │  Warmup steps:    10
║  LR: 2.0e-04  │  Thinking: False
║  LoRA rank: 32  │  Save every: 500 steps
╚══════════════════════════════════════════════════════════╝

⏳ Connecting to Tinker API...


2026-07-13 09:43:06,258 [INFO] ServiceClient initialized for session bb491907-1211-588b-a18f-a0470ba5c63e
2026-07-13 09:43:07,055 [INFO] TrainingClient initialized for model bb491907-1211-588b-a18f-a0470ba5c63e:train:0


✓ Connected!

[Step    0/309] epoch=1/1 lr=0.00e+00 train_nll=0.2483 avg_nll(10)=0.2483 step=23.7s elapsed=24s
[Step    5/309] epoch=1/1 lr=1.00e-04 train_nll=0.1817 avg_nll(10)=0.2016 step=2.4s elapsed=40s
[Step   10/309] epoch=1/1 lr=2.00e-04 train_nll=0.1182 avg_nll(10)=0.2000 step=1.8s elapsed=49s
[Step   15/309] epoch=1/1 lr=1.97e-04 train_nll=0.1970 avg_nll(10)=0.1911 step=1.8s elapsed=58s
[Step   20/309] epoch=1/1 lr=1.93e-04 train_nll=0.1083 avg_nll(10)=0.1578 step=2.1s elapsed=68s
[Step   25/309] epoch=1/1 lr=1.90e-04 train_nll=0.1469 avg_nll(10)=0.1253 step=1.8s elapsed=77s
[Step   30/309] epoch=1/1 lr=1.87e-04 train_nll=0.1275 avg_nll(10)=0.1260 step=1.8s elapsed=86s
[Step   35/309] epoch=1/1 lr=1.83e-04 train_nll=0.1271 avg_nll(10)=0.1396 step=1.8s elapsed=95s
[Step   40/309] epoch=1/1 lr=1.80e-04 train_nll=0.0915 avg_nll(10)=0.1439 step=1.9s elapsed=106s
[Step   45/309] epoch=1/1 lr=1.77e-04 train_nll=0.1013 avg_nll(10)=0.1595 step=1.8s elapsed=115s
[Step   50/309] epoch=1

In [10]:
# ============================================================
# Cell 9: Execution Entrypoint
# ============================================================

# To run SFT training and inference:
# if __name__ == "__main__":
#     if not api_set:
#         print("⚠ Please set your TINKER_API_KEY environment variable in Cell 3 first.")
#         sys.exit(1)
#     
#     


In [11]:
run_test_inference(service_client, sampler_path)

Loading test dataset from: /kaggle/input/datasets/barnobarno/rag-retrievals-ready-for-inference/evaluation_rag_queries_aro_BORO.csv
Loaded 2516 rows for evaluation.
Connecting to Tinker sampler with path: tinker://bb491907-1211-588b-a18f-a0470ba5c63e:train:0/sampler_weights/qwen3_5_sft_final...
Skipped 0 rows (missing/empty context).
Submitting 2516 test inference jobs asynchronously to Tinker...


Running SFT Inference: 100%|██████████| 2516/2516 [00:01<00:00, 1591.69it/s]


FileNotFoundError: [Errno 2] No such file or directory: ''